In [2]:
import pandas as pd
import numpy as np

In [3]:
#parameters
np.random.seed(42)

n_players = 5000
max_days = 30

In [5]:
#generate players and matches data

players = []
matches = []

for player_id in range(1, n_players + 1):
    
    skill = np.random.normal(1000, 200)
    signup_date = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0,10), unit="D")
    
    players.append([player_id, skill, signup_date])
    
    churned = False
    day = 0
    
    early_wins = 0
    early_games = 0
    
    while not churned and day < max_days:
        
        #probability that a player logs in
        play_prob = 0.6 if day < 3 else 0.4
        
        if np.random.rand() < play_prob:
            
            opponent_skill = np.random.normal(1000, 200)
            
            win_prob = 1 / (1 + np.exp((opponent_skill - skill)/200))
            win = np.random.rand() < win_prob
            
            matches.append([
                player_id,
                signup_date + pd.to_timedelta(day, unit="D"),
                skill,
                opponent_skill,
                int(win)
            ])
            
            #tracking early games & wins
            if day < 3:
                early_games += 1
                if win:
                    early_wins += 1
        
        #churn logic
        
        base_churn_prob = 0.02  #baseline churn probability
        
        if early_games > 0:
            early_win_rate = early_wins / early_games
        else:
            early_win_rate = 0.5
        
        #losing early increases churn probability
        churn_prob = base_churn_prob + (0.3 * (1 - early_win_rate))
        
        #capped probability
        churn_prob = min(churn_prob, 0.5)
        
        if np.random.rand() < churn_prob:
            churned = True
        
        day += 1

players = pd.DataFrame(players, columns=[
    "player_id","skill_rating","signup_date"
])

matches = pd.DataFrame(matches, columns=[
    "player_id","match_date","player_skill","opponent_skill","win"
])

In [8]:
#export to csv
players.to_csv("data/players.csv", index=False)
matches.to_csv("data/matches.csv", index=False)